# Week 1 - Data collection, loading and initial inspection

Active scope: the official RouteRanker brief, Week 1. New York is the user-approved alternative to its original Indian-city scope. This notebook does not clean source records, engineer features, run EDA, train models or build an application.

Run `python scripts/week1_inspection.py` first to generate full-table checks. This notebook validates the same frozen inputs, then explains their saved inspection evidence.

In [ ]:
from pathlib import Path
import sys,json,zipfile
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"scripts/week1_inspection.py").exists())
sys.path.insert(0,str(ROOT/"scripts"))
import week1_inspection as week1
print("Project:",ROOT)
print("Active stage: Week 1 only")

## 1. Verify collected source files

Each receipt records its URL/query, retrieval time, bytes and SHA-256. A matching fingerprint establishes file identity, not correctness of the publisher. Downloaded hourly/daily aggregates are exact publisher query responses, not ticket-level records.

In [ ]:
summary=json.loads((week1.REPORT/"inspection_summary.json").read_text())
expected={row["local_path"]:row["sha256"] for row in pd.read_csv(week1.REPORT/"source_inventory.csv").to_dict("records")}
receipts=week1.inventory()
actual={row["local_path"]:row["sha256"] for row in receipts}
assert actual==expected,"Inputs changed: rerun the full Week 1 inspection before using these reports."
print("Verified source receipts:",len(receipts))
display(pd.DataFrame(receipts)[["local_path","bytes","sha256_verified"]].head(8))

## 2. Load the essential GTFS files

Read the original ZIP directly into pandas; extraction to a separate text folder is unnecessary. IDs are deliberately kept as strings. The five previews below are bounded; the saved inspection reports cover all rows, not just these previews.

In [ ]:
source=week1.RAW/"gtfs/current/manhattan.zip"
with zipfile.ZipFile(source) as archive:
    routes=pd.read_csv(archive.open("routes.txt"),dtype="string")
    trips_preview=pd.read_csv(archive.open("trips.txt"),dtype="string",nrows=5)
    stop_times_preview=pd.read_csv(archive.open("stop_times.txt"),dtype="string",nrows=5)
    stops_preview=pd.read_csv(archive.open("stops.txt"),dtype="string",nrows=5)
    calendar=pd.read_csv(archive.open("calendar.txt"),dtype="string")
print("Manhattan catalogue routes:",routes.shape)
routes.info(memory_usage="deep")
for name,frame in [("routes",routes.head()),("trips",trips_preview),("stop_times",stop_times_preview),("stops",stops_preview),("calendar",calendar.head())]:
    print(name)
    display(frame)

## 3. Dataset coverage and dimensions

Different borough feeds can share route catalogues. Catalogue rows do not equal active routes. Trip rows are schedule templates, not passengers or automatically daily trip counts. Calendar dates define planned applicability, not observed operations.

In [ ]:
inventory=pd.read_csv(week1.REPORT/"gtfs_inventory.csv")
display(inventory[["feed","routes","trips","stop_times","stops","calendar","routes_referenced_by_trips","calendar_start","calendar_end"]])
display(pd.DataFrame([summary["hourly_source"],summary["daily_source"]])[["resolution","rows","columns","dates","published_route_identifiers","first_date","last_date"]])

## 4. Initial quality checks

Inspect missingness, loaded types, memory, duplicates, keys, references and basic formats. Missing optional fields are reported rather than removed. Coordinate ranges do not prove road/route correctness. GTFS times may exceed 24:00:00.

In [ ]:
folder=week1.REPORT/"gtfs/current_manhattan"
display(pd.read_csv(folder/"table_inspection.csv"))
display(pd.read_csv(folder/"column_inspection.csv"))
display(pd.read_csv(folder/"key_and_field_checks.csv"))

## 5. Ridership source inspection and limitations

The hourly pilot covers three months; the additional collected history is daily. Electronic-fare estimates may contain publisher imputation and exclude cash/free travel. Transfers are already included. Separate publisher daily queries verify extraction arithmetic, not true passenger totals. Original fare-category checks cover only a bounded sample.

In [ ]:
display(pd.read_csv(week1.REPORT/"hourly_source_column_inspection.csv"))
display(pd.read_csv(week1.REPORT/"daily_source_column_inspection.csv"))
print("Bounded raw sample:")
print(json.dumps(summary["bounded_raw_sample"],indent=2))
print(summary["hourly_source"]["reconciliation"])
print(summary["daily_source"]["reconciliation"])

## Week 1 handoff

Collection, essential-file loading and initial inspection are complete for the approved New York scope. The original Lepton/23-Indian-city scope is not fulfilled. No cleaning, route-frequency/duration features, EDA, scaling, models, scenarios, database/API or dashboard progress is claimed here.

Read `reports/week1/WEEK_1_REPORT.md`. Week 2 remains a separate next step; Streamlit is the fixed Week 7 dashboard, with FastAPI/PostgreSQL allowed later.